# Customer Complaint Classification — Phase 1 & 2 Starter Notebook
### "Is Reasoning Worth the Cost?" Mini Project

This notebook covers:
1. Setting up API access to an LLM
2. Testing your first API call
3. Loading and exploring the complaint dataset
4. Cleaning and preparing a balanced sample
5. Building the baseline **zero-shot** classification pipeline
6. Testing it and logging results

**How to use this notebook:** Run each cell in order, top to bottom (click the ▶ button on the left of each cell, or press Shift+Enter). Read the markdown notes above each code cell before running it.


## Step 0 — Install required packages
Run this once per Colab session.

In [1]:
!pip install openai pandas scikit-learn -q
print("Packages installed.")
!pip install pyarrow -q

Packages installed.


## Step 1 — Set up your FREE API key (Groq)

We're using the **free Groq API** — no credit card, no verification hoops. Groq runs open models (like Llama 3.3) at very high speed, with a generous free daily limit (thousands of requests/day) — plenty for this project.

1. Go to **console.groq.com**, sign in (Google login works).
2. Go to **API Keys** → **Create API Key**.
3. Copy the key and paste it below.

**Important:** the key must be wrapped in quote marks in the code, like `API_KEY = "your-key-here"` — without the quotes, Python tries to run it as code instead of treating it as text, which causes a `NameError`.

**Never commit your real API key to GitHub.** For now, paste it directly below (fine for solo testing). Later, use Colab's "Secrets" manager (🔑 icon in the left sidebar) so it's not visible in the notebook itself.


In [ ]:
from openai import OpenAI

# Option A (quick, for now): paste your key directly, WITH the quote marks around it
API_KEY = "PASTE_YOUR_KEY_HERE"   # <-- keep the quotes! this must be a text string

# Option B (safer, recommended once it's working): use Colab Secrets
# from google.colab import userdata
# API_KEY = userdata.get('GROQ_API_KEY')

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=API_KEY,
)
MODEL_NAME = "openai/gpt-oss-120b"
print("Client ready.")

Client ready.


In [9]:
from google.colab import userdata
API_KEY = userdata.get('GROQ_API_KEY')

## Step 2 — Test call: make sure everything works end to end

In [3]:
test_complaint = "I was charged twice for my subscription this month and support has not responded."
categories = ["Billing", "Product Defect", "Customer Service", "Fraud", "Other"]

prompt = f'Classify this customer complaint into exactly one of these categories: {categories}.\n\nComplaint: "{test_complaint}"\n\nReply with ONLY the category name, nothing else.'

response = client.chat.completions.create(
    model=MODEL_NAME,
    max_tokens=200,                 # raised: reasoning models need budget for thinking + answer
    reasoning_effort="low",         # keep thinking minimal for a simple classification task
    messages=[{"role": "user", "content": prompt}]
)
print("Model said:", response.choices[0].message.content.strip())

Model said: Billing


**Checkpoint:** if you saw a category printed above (like `Billing`), your API setup works. Don't move on until this works for everyone on the team.

## Step 3 — Load the dataset

1. Download the dataset from Kaggle: search **"Consumer Complaints Dataset for NLP"** (CFPB-based).
2. In Colab, click the folder icon on the left sidebar → upload icon → upload the CSV file.
3. Update the filename below to match exactly what you uploaded.

The CFPB dataset usually has columns like `Product` (the category) and `Consumer complaint narrative` (the complaint text) — if your column names differ, adjust the `TEXT_COL` and `LABEL_COL` variables below after checking `df.columns`.


In [20]:
!ls

rows.csv  sample_data


In [4]:
import csv
import pandas as pd

FILENAME = "/content/rows.csv"   # adjust to your fresh upload's exact name

rows = []
bad_row_count = 0

with open(FILENAME, "r", encoding="utf-8", errors="replace") as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            rows.append({
                "Product": row.get("Product"),
                "Consumer complaint narrative": row.get("Consumer complaint narrative"),
            })
        except Exception:
            bad_row_count += 1

df = pd.DataFrame(rows)
print("Total rows loaded:", len(df))
print("Rows that failed to parse:", bad_row_count)

has_text = df["Consumer complaint narrative"].notna().sum()
print("Rows with narrative text:", has_text)

Total rows loaded: 56184
Rows that failed to parse: 0
Rows with narrative text: 56184


In [5]:
print(df["Product"].value_counts())

Product
Credit reporting, credit repair services, or other personal consumer reports    27276
Debt collection                                                                 10267
Credit card or prepaid card                                                      5108
Mortgage                                                                         4670
Checking or savings account                                                      4024
Student loan                                                                     1890
Vehicle loan or lease                                                            1160
Money transfer, virtual currency, or money service                                953
Payday loan, title loan, or personal loan                                         836
Name: count, dtype: int64


In [6]:
CATEGORY_MAP = {
    "Credit reporting, credit repair services, or other personal consumer reports": "Credit Reporting",
    "Credit reporting": "Credit Reporting",
    "Debt collection": "Debt Collection",
    "Mortgage": "Mortgage",
    "Credit card or prepaid card": "Credit Card",
    "Credit card": "Credit Card",
    "Bank account or service": "Bank Account/Service",
    "Checking or savings account": "Bank Account/Service",
}

df["clean_category"] = df["Product"].map(CATEGORY_MAP)
df_clean = df.dropna(subset=["clean_category"]).copy()
df_clean = df_clean[df_clean["Consumer complaint narrative"].str.len().between(20, 500)]

print("Rows after cleaning:", len(df_clean))
print(df_clean["clean_category"].value_counts())

Rows after cleaning: 1644
clean_category
Credit Reporting        935
Debt Collection         508
Credit Card              93
Bank Account/Service     63
Mortgage                 45
Name: count, dtype: int64


In [7]:
TEXT_COL = "Consumer complaint narrative"

PER_CATEGORY_DESIGN = 4
PER_CATEGORY_TEST = 35     # lowered to fit your smallest category (Mortgage, 45 rows)

design_rows, test_rows = [], []
for cat, group in df_clean.groupby("clean_category"):
    group = group.sample(frac=1, random_state=42)
    design_rows.append(group.iloc[:PER_CATEGORY_DESIGN])
    remaining = group.iloc[PER_CATEGORY_DESIGN:]
    test_rows.append(remaining.iloc[:PER_CATEGORY_TEST])

design_df = pd.concat(design_rows).reset_index(drop=True)
test_df = pd.concat(test_rows).reset_index(drop=True)

design_df.to_csv("prompt_design_set.csv", index=False)
test_df.to_csv("test_set.csv", index=False)

print("Design set:", design_df.shape)
print("Test set:", test_df.shape)
print(test_df["clean_category"].value_counts())

from google.colab import files
files.download("prompt_design_set.csv")
files.download("test_set.csv")

Design set: (20, 3)
Test set: (175, 3)
clean_category
Bank Account/Service    35
Credit Card             35
Credit Reporting        35
Debt Collection         35
Mortgage                35
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

# List files in the /content/ directory to help confirm the file's presence and exact name
print("Files in /content/:")
!ls -F /content/

Files in /content/:
sample_data/


In [ ]:
import os

file_to_check = FILENAME # Using the FILENAME variable from the previous cell

if os.path.exists(file_to_check):
    print(f"The file '{file_to_check}' exists in the current directory.")
else:
    print(f"The file '{file_to_check}' does NOT exist in the current directory.")
    print("Please ensure you have uploaded the correct file and that its name matches the FILENAME variable.")

The file '/content/rows.csv.xlsx' does NOT exist in the current directory.
Please ensure you have uploaded the correct file and that its name matches the FILENAME variable.


## Step 4 — Explore the data
Check category names and how balanced they are.

In [23]:
TEXT_COL = "Consumer complaint narrative"   # <-- update if different
LABEL_COL = "Product"                          # <-- update if different

print(df[LABEL_COL].value_counts().head(20))

Product
Credit reporting, credit repair services, or other personal consumer reports    23220
Debt collection                                                                  8453
Credit card or prepaid card                                                      4329
Mortgage                                                                         3949
Checking or savings account                                                      3505
Student loan                                                                     1542
Vehicle loan or lease                                                             989
Money transfer, virtual currency, or money service                                810
Payday loan, title loan, or personal loan                                         722
Name: count, dtype: int64


## Step 5 — Clean and narrow down to categories

CFPB data often has many overlapping product categories (e.g., "Credit card", "Credit card or prepaid card"). Pick **5 clear, distinct categories** for your project (matching the plan) and map similar raw labels into them.

Edit the `CATEGORY_MAP` dictionary below based on what you saw in Step 4's output — the keys are the RAW category names from the dataset, and the values are your 5 clean category names.


In [24]:
CATEGORY_MAP = {
    "Credit reporting, credit repair services, or other personal consumer reports": "Credit Reporting",
    "Credit reporting": "Credit Reporting",
    "Debt collection": "Debt Collection",
    "Mortgage": "Mortgage",
    "Credit card or prepaid card": "Credit Card",
    "Credit card": "Credit Card",
    "Bank account or service": "Bank Account/Service",
    "Checking or savings account": "Bank Account/Service",
}

df["clean_category"] = df["Product"].map(CATEGORY_MAP)
df_clean = df.dropna(subset=["clean_category"]).copy()
df_clean = df_clean[df_clean["Consumer complaint narrative"].str.len().between(20, 500)]

print("Rows after cleaning:", len(df_clean))
print(df_clean["clean_category"].value_counts())

Rows after cleaning: 635
clean_category
Credit Reporting        356
Debt Collection         208
Credit Card              35
Bank Account/Service     20
Mortgage                 16
Name: count, dtype: int64


## Step 6 — Build a balanced sample and split into two sets

- **Prompt-design set** (~20 examples): used to build/refine your prompts. You'll look at these closely.
- **Test set** (the rest, e.g. 200 per category = 1000 total): used ONLY for final results. Don't peek at this while designing prompts — that would bias your results.


In [ ]:
PER_CATEGORY_DESIGN = 4      # ~20 total across 5 categories
PER_CATEGORY_TEST = 60       # adjust based on budget/time; 60 x 5 = 300 total test cases

design_rows, test_rows = [], []

for cat, group in df_clean.groupby("clean_category"):
    group = group.sample(frac=1, random_state=42)  # shuffle
    design_rows.append(group.iloc[:PER_CATEGORY_DESIGN])
    remaining = group.iloc[PER_CATEGORY_DESIGN:]
    test_rows.append(remaining.iloc[:PER_CATEGORY_TEST])

design_df = pd.concat(design_rows).reset_index(drop=True)
test_df = pd.concat(test_rows).reset_index(drop=True)

print("Prompt-design set size:", len(design_df))
print("Test set size:", len(test_df))

design_df.to_csv("prompt_design_set.csv", index=False)
test_df.to_csv("test_set.csv", index=False)
print("\nSaved: prompt_design_set.csv and test_set.csv")
print("Download these from the Colab file browser and add them to your GitHub repo's /data folder.")

## Step 7 — Build the baseline zero-shot classification function

This is the simplest possible version: one plain instruction, no examples, no reasoning steps. This becomes your **baseline** — everything else (few-shot, CoT, self-consistency) will be compared against this.


In [ ]:
import time

CATEGORIES = sorted(df_clean["clean_category"].unique().tolist())

def classify_zero_shot(complaint_text):
    prompt = (
        f"Classify this customer complaint into exactly one of these categories: {CATEGORIES}.\n\n"
        f'Complaint: "{complaint_text}"\n\n'
        f"Reply with ONLY the category name, nothing else."
    )
    start = time.time()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        max_tokens=200,              # raised: reasoning models need budget for thinking + answer
        reasoning_effort="low",      # keep thinking minimal for a simple classification task
        messages=[{"role": "user", "content": prompt}]
    )
    elapsed = time.time() - start
    prediction = response.choices[0].message.content.strip()
    tokens_used = response.usage.total_tokens
    return prediction, elapsed, tokens_used

# Quick test on one example
sample_text = design_df.iloc[0][TEXT_COL]
true_label = design_df.iloc[0]["clean_category"]
pred, t, tok = classify_zero_shot(sample_text)
print("Complaint:", sample_text[:150], "...")
print("True label:", true_label)
print("Predicted:", pred, "| Time:", round(t,2), "s | Tokens:", tok)

## Step 8 — Run the baseline on the prompt-design set and check accuracy

This is just a small sanity check (20 examples) before running the full experiment later. If accuracy looks reasonable here (not near-random), your pipeline is working correctly.


In [ ]:
results = []

for idx, row in design_df.iterrows():
    pred, elapsed, tokens = classify_zero_shot(row[TEXT_COL])
    results.append({
        "complaint": row[TEXT_COL],
        "true_label": row["clean_category"],
        "predicted": pred,
        "correct": pred.strip().lower() == row["clean_category"].strip().lower(),
        "time_sec": elapsed,
        "tokens": tokens,
    })
    print(f"[{idx+1}/{len(design_df)}] True: {row['clean_category']:20s} | Predicted: {pred}")

results_df = pd.DataFrame(results)
accuracy = results_df["correct"].mean()
print(f"\nZero-shot accuracy on prompt-design set: {accuracy:.2%}")
print(f"Average time per call: {results_df['time_sec'].mean():.2f}s")
print(f"Average tokens per call: {results_df['tokens'].mean():.1f}")

results_df.to_csv("baseline_zero_shot_design_results.csv", index=False)
print("\nSaved: baseline_zero_shot_design_results.csv")

## ✅ Checkpoint — What you've accomplished

- Confirmed API access works
- Loaded and cleaned the real dataset
- Built a balanced prompt-design set and test set
- Built and tested your first working classification function (zero-shot baseline)
- Got a first accuracy number

## Next steps (Phase 3)

Next, we'll build the other 3 prompting conditions on top of this same structure:
- **Few-shot** (add solved examples before the question)
- **Chain-of-Thought** (ask the model to explain its reasoning before answering)
- **Self-consistency** (ask multiple times, take the majority vote)

Come back with this notebook once Step 8 is working and showing a reasonable accuracy number, and we'll build those next together.


---
# Phase 3 — Building the Other 3 Prompting Conditions

Your zero-shot baseline is working (85% on the design set). Now let's add the other 3 conditions: **few-shot**, **chain-of-thought (CoT)**, and **self-consistency**. Each builds on the same pattern as `classify_zero_shot`.


## Few-shot: show the model a few solved examples first

We'll hand-pick 1 example per category from the prompt-design set to use as the "teaching examples," then classify everything else (never reusing a complaint as both an example and a test case).


In [ ]:
# Pick one example per category from the design set to use as few-shot examples
FEW_SHOT_EXAMPLES = design_df.groupby("clean_category").first().reset_index()
print(FEW_SHOT_EXAMPLES[["clean_category", TEXT_COL]])

In [ ]:
def build_few_shot_block():
    lines = []
    for _, row in FEW_SHOT_EXAMPLES.iterrows():
        lines.append(f'Complaint: "{row[TEXT_COL]}"\nCategory: {row["clean_category"]}\n')
    return "\n".join(lines)

FEW_SHOT_BLOCK = build_few_shot_block()

def classify_few_shot(complaint_text):
    prompt = (
        f"Here are some examples of customer complaints and their correct category:\n\n"
        f"{FEW_SHOT_BLOCK}\n"
        f"Now classify this new complaint into exactly one of these categories: {CATEGORIES}.\n\n"
        f'Complaint: "{complaint_text}"\n\n'
        f"Reply with ONLY the category name, nothing else."
    )
    start = time.time()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        max_tokens=200,
        reasoning_effort="low",
        messages=[{"role": "user", "content": prompt}]
    )
    elapsed = time.time() - start
    prediction = response.choices[0].message.content.strip()
    tokens_used = response.usage.total_tokens
    return prediction, elapsed, tokens_used

# Quick test
pred, t, tok = classify_few_shot(design_df.iloc[10][TEXT_COL])
print("Predicted:", pred, "| True:", design_df.iloc[10]["clean_category"], "| Time:", round(t,2), "| Tokens:", tok)

## Chain-of-Thought (CoT): ask the model to explain its reasoning first

This is also where we capture the model's **explanation** — you'll need this later for the faithfulness check (Phase 5).


In [ ]:
def classify_cot(complaint_text):
    prompt = (
        f"Classify this customer complaint into exactly one of these categories: {CATEGORIES}.\n\n"
        f'Complaint: "{complaint_text}"\n\n'
        f"First, think step by step about which category fits best and briefly explain your reasoning in 1-2 sentences. "
        f"Then, on a new final line, write EXACTLY: 'Final Category: <category name>'."
    )
    start = time.time()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        max_tokens=300,
        reasoning_effort="low",
        messages=[{"role": "user", "content": prompt}]
    )
    elapsed = time.time() - start
    full_text = response.choices[0].message.content.strip()
    tokens_used = response.usage.total_tokens

    # Extract the final category from the last line
    prediction = None
    for line in full_text.splitlines():
        if line.strip().lower().startswith("final category:"):
            prediction = line.split(":", 1)[1].strip()
    if prediction is None:
        prediction = full_text.strip().splitlines()[-1]  # fallback: last line

    explanation = full_text  # keep the full text for the faithfulness check later
    return prediction, explanation, elapsed, tokens_used

# Quick test
pred, expl, t, tok = classify_cot(design_df.iloc[10][TEXT_COL])
print("Predicted:", pred, "| True:", design_df.iloc[10]["clean_category"])
print("Explanation:", expl)
print("Time:", round(t,2), "| Tokens:", tok)

## Self-consistency: ask multiple times, take the majority vote

This calls `classify_cot` several times (default 5) for the same complaint and picks the most common final answer. This is the most expensive condition — it costs roughly N times as much as a single CoT call.


In [ ]:
from collections import Counter

def classify_self_consistency(complaint_text, n_samples=5):
    predictions = []
    explanations = []
    total_time = 0
    total_tokens = 0

    for _ in range(n_samples):
        pred, expl, t, tok = classify_cot(complaint_text)
        predictions.append(pred)
        explanations.append(expl)
        total_time += t
        total_tokens += tok

    vote_counts = Counter(predictions)
    majority_prediction = vote_counts.most_common(1)[0][0]

    return majority_prediction, explanations, total_time, total_tokens, vote_counts

# Quick test (this takes a few seconds since it makes 5 calls)
pred, expls, t, tok, votes = classify_self_consistency(design_df.iloc[10][TEXT_COL], n_samples=5)
print("Majority vote:", pred, "| True:", design_df.iloc[10]["clean_category"])
print("Vote breakdown:", dict(votes))
print("Total time:", round(t,2), "| Total tokens:", tok)

## ✅ Checkpoint — All 4 conditions built

You now have 4 working functions:
- `classify_zero_shot(text)` → prediction, time, tokens
- `classify_few_shot(text)` → prediction, time, tokens
- `classify_cot(text)` → prediction, explanation, time, tokens
- `classify_self_consistency(text, n_samples)` → majority prediction, explanations, total time, total tokens, vote breakdown

## Next: Phase 4 — run all 4 conditions on the full test set

Before running on all 300 test cases (expensive — self-consistency alone makes 5×300 = 1500 calls), first test all 4 functions on just the 20-example prompt-design set to sanity check accuracy and catch any bugs cheaply. Come back once you've run these quick tests and we'll build the full experiment-logging loop for Phase 4 together.
